In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

import os

import pandas as pd 

from time import sleep

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from pandas import ExcelWriter

import datetime



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BE BNBE' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.9")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 

            'BNBE BE 1':'https://www.nbb.be/doc/be/be/protocol/current_codes.xlsx', 

            'BNBE BE 2':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-credit/listes-3', 

            'BNBE BE 3':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-credit/listes-0', 

            'BNBE BE 4':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-credit/listes-1', 

            'BNBE BE 5':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-credit/listes-6', 

            'BNBE BE 6':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/entreprises-dassurance-et-de-24', 

            'BNBE BE 7':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/entreprises-dassurance-et-de-15', 

            'BNBE BE 8':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/entreprises-dassurance-et-de-16', 

            'BNBE BE 9':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/entreprises-dassurance-et-de-17', 

            'BNBE BE 11':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/societes-de-bourse/listes/societes-0', 

            'BNBE BE 12':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/societes-de-bourse/listes-3', 

            'BNBE BE 13':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-15', 

            'BNBE BE 14':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-7', 

            'BNBE BE 15':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-8', 

            'BNBE BE 16':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-9', 

            'BNBE BE 17':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-11', 

            'BNBE BE 18':'https://www.nbb.be/fr/supervision-financiere/controle-prudentiel/domaines-de-controle/etablissements-de-paiement-et-12'

            }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



RegulationType = {1 : "Licensed", 2 : "Licensed", 3 : "Registered", 4 : "EEA Authorised", 5 : "Registered", 6 : "Licensed", 7 : "EEA Authorised", 8 : "EEA Authorised", 9 : "Licensed", 11 : "Licensed", 12 : "Registered", 13 : "Licensed", 14 : "Registered", 15 : "EEA Authorised", 16 : "Licensed", 17 : "Registered", 18 : "EEA Authorised"}

processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,'//*[@id="popup-buttons"]/button[1]').click()

    except Exception as err:

        print('[ERROR] : Failed to click "Allow all" button on the cookies banner:', err)



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):



    index_reg = int(reg.split(' ')[-1])

    driver.get(regdict[reg])

    sleep(3)

 

    if reg == 'BNBE BE 1':

        file =  check_dowload_files(tempfolder, "csv" )

        filePath = os.path.join(tempfolder, file)

        columns = ['From', 'To', 'Biccode', 'NameDutch', 'NameFrench', 'NameGerman', 'NameEnglish']

        df = pd.read_excel(filePath)

        df.columns = columns

        df = df[1:]

        df = df.reset_index(drop=True)



        print(f"[INFO] : Working {k+1}/{len(regdict)} {reg} | len company = {len(df)}")

        for index, row in df.iterrows() :

            name = ''

            if (str(row['NameDutch']) != 'nan') and (str(row['NameDutch']) != 'VRIJ') and (str(row['NameDutch']) != '-'):

                name = str(row['NameDutch'])

            if (str(row['NameFrench']) != 'nan') and (str(row['NameFrench']) != 'LIBRE') :

                if len(name) > 0 :

                    name = name + " | " + str(row['NameFrench'])

                else:

                    name = str(row['NameFrench'])

            if str(row['NameGerman']) != 'nan':

                name = name + " | " + str(row['NameGerman'])

            if str(row['NameEnglish']) != 'nan':

                name = name + " | " + str(row['NameEnglish'])



            if len(name) > 0 :

                if (str(row['Biccode']) != 'NAV') :

                    sqldict['InternalID_1'].append(str(row['Biccode']))

                    sqldict['InternalID_1_type'].append('Bic code')

                else:

                    sqldict['InternalID_1'].append('')

                    sqldict['InternalID_1_type'].append('')



                sqldict['Name'].append(name)

                sqldict['ListProcessDate'].append(processdate)

                sqldict['RegCtry'].append(reg.split(' ')[0]) 

                sqldict['RegCode'].append(reg.split(' ')[1])

                sqldict['ListCode'].append(reg.split(' ')[-1])

                sqldict['RegulationType'].append(RegulationType[index_reg])

       

        sqldict = bourange_same_length_array(sqldict)

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))



    else:



        soup = BeautifulSoup(driver.page_source, "html.parser")

        if soup.find('div', {'id':'popup-buttons'}):

            click_on_cookies(driver)

            sleep(1)

        

        for time in range(10):

            try:

                soup = BeautifulSoup(driver.page_source, "html.parser")

                Warning_message = soup.find("div",{"id":"main-content-inner"}).find("div",{"class":"alert alert-block alert-warning messages warning"})

                if Warning_message == None :

                    break

            except:

                print(f"[INFO] : The webservice returned an error. {time}/10 try refreshing)")

                driver.refresh()

                sleep(2)      

        else:

            raise Exception(f'[ERROR] : Run Script again.\nWarning message \n{Warning_message}')

        

        mainInfo_ul = soup.find("div",{"id":"PrudentialList"}).find("ul",{"class":"List1"})

        companies = []



        if reg == 'BNBE BE 2':

            ul_s = mainInfo_ul.find_all("ul")

            for i, ul in enumerate(ul_s):

                if len(ul.attrs) == 0 :

                    companies = companies + ul.find_all("li")



        elif reg == 'BNBE BE 5' or reg == 'BNBE BE 11':

            tr_s = []

            for tbody in mainInfo_ul.find_all("tbody"):

                tr_s = tr_s + tbody.find_all("tr") 

            for i, tr in enumerate(tr_s):

                if len(tr.attrs) == 0 :

                    companies.append(tr.find_all("td")[0])

        else:

            try:

                tr_s = mainInfo_ul.find("tbody").find_all("tr")

            except:

                tr_s = []

            for i, tr in enumerate(tr_s):

                if len(tr.attrs) == 0 :

                    companies.append(tr.find_all("td")[0])

        

        print(f"[INFO] : Working {k+1}/{len(regdict)} {reg} | len company = {len(companies)}")

        for l, companie in enumerate(companies):

            info = companie.text.replace('\t', '').replace('\xa0', '')

            companie_name = companie.find("strong").text

            preview_adr = 'Numéro'  if info.find('Numéro') != -1 else 'Date'

            

            try:

                company_type = companie.find("em").text

            except:

                company_type = ''



            try:

                NumID = aa.split('unique :')[1].split('Date')[0].replace('\n', '').strip()

                sqldict['InternalID_1'].append(NumID)

                sqldict['InternalID_1_type'].append('ID number')

            except:

                sqldict['InternalID_1'].append('')

                sqldict['InternalID_1_type'].append('')



            try:

                addr = info.split(preview_adr)[0].replace(companie_name, '').replace(company_type, '').strip()

                Address_1 = addr.split('\n')[0]

                if len(addr.split('\n'))>2 :

                    Zip = addr.split('\n')[2].split(' ')[0]

                    City = addr.split('\n')[2].split(' ')[-1]     

                    Address_2 = addr.split('\n')[1]

                else:

                    Zip = addr.split('\n')[1].split(' ')[0]

                    City = addr.split('\n')[1].split(' ')[-1]

                    Address_2 = ''

            except:

                Address_1 = Zip = City = Address_2 = ''



            sqldict['Name'].append(companie_name)

            sqldict['EntryType'].append(company_type)   # PAS tres sure de l'info

            sqldict['Address_1'].append(Address_1)

            sqldict['Address_2'].append(Address_2)

            sqldict['City'].append(City)

            sqldict['Zip'].append(Zip)

            sqldict['Cntry'].append("BE")

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])

            sqldict['RegulationType'].append(RegulationType[index_reg])



        sqldict = bourange_same_length_array(sqldict)



# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)



# drop duplicates and Revoked company

df = df.drop_duplicates(subset = ["Name", "InternalID_1"], keep = 'first')

df = df.reset_index(drop=True)



df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    